# Train efficiently, keep the evidence · 29 October 2026
60-minute core session. Real **Understanding Clouds from Satellite Images** data,
four independent targets (Fish, Flower, Gravel, Sugar). This is a small training
mechanics lesson, not a competition solution or a promised speedup.

Run Setup first. Later slide links identify the section and cell to revisit.
The notebook writes its complete, editable runtime to `cloud_training.py` beside
this notebook. The code is embedded below, so no project checkout is required.

Download the 384×576 mirror yourself, or explicitly select original 2100×1400
files. Never mix CSV/RLE variants. No raw data or synthetic fallback is bundled.
A `CLOUD_SPLIT_MANIFEST` can reuse bootcamp `session1_evidence.json` image splits.

Hardware: FP32 works on CPU; BF16 CPU autocast demonstrates execution only.
CUDA FP16 uses GradScaler; native CUDA BF16 uses no scaler in this lesson.
Unsupported CUDA/precision requests fail clearly. Keep outputs private if they
contain downloaded images or fitted results. [Study guide](https://manyakaistha.github.io/DLGenAI-26T3/session-4/).


## 00 · Setup — run these cells first
Use the existing compatible PyTorch runtime if available. If dependencies are
missing, run the next installation cell, restart after replacing imported binary
packages, then run the runtime and dataset cells. Network access is needed for
installation and your separate Kaggle download, not for pretrained weights.


In [ ]:
# Run only if this runtime needs the teaching dependencies.
# %pip install torch==2.8.0 numpy==2.2.6 Pillow==12.3.0


In [ ]:
from pathlib import Path
Path('cloud_training.py').write_text('# /// script\n# requires-python = ">=3.10"\n# dependencies = ["torch==2.8.0", "numpy==2.2.6", "Pillow==12.3.0"]\n# ///\n"""Clouds teaching runtime: real-data AMP, accumulation and epoch-boundary resume.\nNo industrial data, pretrained download, leaderboard submission or synthetic fallback.\n"""\nfrom __future__ import annotations\nimport copy, csv, hashlib, json, math, os, random, time\nfrom contextlib import nullcontext\nfrom dataclasses import asdict, dataclass\nfrom itertools import islice\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nCLASSES = (\'Fish\',\'Flower\',\'Gravel\',\'Sugar\')\nSIZES = {\'resized_384x576\': (576,384), \'original\': (2100,1400)}\n\n@dataclass\nclass Config:\n    data_root: str = \'data/bootcamp/clouds_384x576\'\n    variant: str = \'resized_384x576\'\n    output: str = \'runs/cloud_efficiency\'\n    precision: str = \'fp32\'\n    device: str = \'auto\'\n    seed: int = 29\n    height: int = 64\n    width: int = 96\n    batch: int = 2\n    accumulation: int = 2\n    epochs: int = 3\n    train_cap: int = 32\n    val_cap: int = 16\n    lr: float = .001\n    warmup_updates: int = 2\n    patience: int = 5\n    smoke_ids: str = \'\'\n    split_manifest: str = \'\'\n\n    @classmethod\n    def from_env(cls):\n        c=cls()\n        for field,key in [(\'data_root\',\'CLOUD_DATA_ROOT\'),(\'variant\',\'CLOUD_DATA_VARIANT\'),\n                          (\'output\',\'CLOUD_OUTPUT_ROOT\'),(\'precision\',\'PRECISION\'),\n                          (\'device\',\'TRAIN_DEVICE\'),(\'smoke_ids\',\'CLOUD_SMOKE_IMAGE_IDS\'),\n                          (\'split_manifest\',\'CLOUD_SPLIT_MANIFEST\')]:\n            setattr(c,field,os.environ.get(key,getattr(c,field)))\n        for field in [\'seed\',\'height\',\'width\',\'batch\',\'accumulation\',\'epochs\',\'train_cap\',\'val_cap\',\'warmup_updates\',\'patience\']:\n            setattr(c,field,int(os.environ.get(field.upper(),getattr(c,field))))\n        return c\n\n    def validate(self):\n        if self.variant not in SIZES: raise ValueError(\'Choose original or resized_384x576, using matching CSV and images.\')\n        if self.precision not in (\'fp32\',\'fp16\',\'bf16\'): raise ValueError(\'PRECISION must be fp32, fp16 or bf16\')\n        if min(self.batch,self.accumulation,self.epochs,self.height,self.width,self.patience)<1: raise ValueError(\'Batch, accumulation, epochs, geometry and patience must be positive\')\n        if self.width*2 != self.height*3: raise ValueError(\'Preserve the cloud 3:2 width:height aspect ratio (e.g. H64/W96).\')\n        if min(self.train_cap,self.val_cap,self.warmup_updates)<0: raise ValueError(\'Caps and warmup cannot be negative\')\n        return self\n\n\ndef rle_runs(text, pixels):\n    if not text.strip(): return []\n    tokens=text.split()\n    if len(tokens)%2: raise ValueError(\'RLE needs start/length pairs\')\n    try: values=[int(v) for v in tokens]\n    except ValueError as e: raise ValueError(\'RLE needs integer tokens\') from e\n    runs=[];previous=0\n    for start,length in zip(values[::2],values[1::2]):\n        end=start+length-1\n        if start<1 or length<1 or end>pixels or start<=previous:\n            raise ValueError(\'RLE outside canvas, nonpositive, unsorted or overlapping\')\n        runs.append((start-1,end));previous=end\n    return runs\n\n\ndef decode(text, size_wh):\n    w,h=size_wh;flat=np.zeros(w*h,np.uint8)\n    for start,end in rle_runs(text,w*h): flat[start:end]=1\n    return flat.reshape((h,w),order=\'F\')\n\n\ndef prepare(c):\n    c.validate();root=Path(c.data_root).expanduser();csv_path=root/\'train.csv\'\n    if not csv_path.is_file() or not (root/\'train_images\').is_dir():\n        raise FileNotFoundError(f\'{root}: download the selected real Clouds variant; expected train.csv and train_images/. No synthetic fallback.\')\n    records={};max_end=0;size=SIZES[c.variant]\n    with csv_path.open(newline=\'\') as f:\n        reader=csv.DictReader(f)\n        if not {\'Image_Label\',\'EncodedPixels\'}.issubset(reader.fieldnames or []): raise ValueError(\'Unexpected cloud CSV schema\')\n        for row in reader:\n            name,label=row[\'Image_Label\'].rsplit(\'_\',1)\n            if Path(name).name!=name or label not in CLASSES: raise ValueError(\'Bad image/category key\')\n            entry=records.setdefault(name,{})\n            if label in entry: raise ValueError(\'Duplicate image/category row\')\n            text=row[\'EncodedPixels\'].strip();runs=rle_runs(text,size[0]*size[1])\n            if runs:max_end=max(max_end,runs[-1][1])\n            entry[label]=text\n    if any(set(v)!=set(CLASSES) for v in records.values()): raise ValueError(\'Every image needs all four category rows; empty RLE is allowed\')\n    if c.variant==\'original\' and max_end<=576*384: raise ValueError(\'CSV coordinates look like the mirror; do not mix source variants\')\n    names=sorted(records);smoke=[x.strip() for x in c.smoke_ids.split(\',\') if x.strip()]\n    if smoke:\n        if len(smoke)!=len(set(smoke)) or not set(smoke)<=set(names): raise ValueError(\'Smoke IDs must be distinct IDs present in CSV\')\n        names=sorted(smoke)\n    if c.split_manifest:\n        old=json.loads(Path(c.split_manifest).read_text())\n        if old[\'dataset_variant\']!=c.variant:raise ValueError(\'Saved split variant differs\')\n        train=list(old[\'train_image_ids\']);val=list(old[\'validation_image_ids\'])\n        if set(train)&set(val) or len(set(train))!=len(train) or len(set(val))!=len(val): raise ValueError(\'Split overlaps or contains duplicates\')\n        if not set(train+val)<=set(names): raise ValueError(\'Saved split IDs absent from selected data\')\n    else:\n        if len(names)<4:raise ValueError(\'At least four images required for the teaching holdout\')\n        rng=random.Random(c.seed);rng.shuffle(names);n=max(1,round(.2*len(names)))\n        val=names[:n];train=names[n:]\n    # Caps are applied only after assigning image IDs to disjoint partitions.\n    train=train[:c.train_cap or None];val=val[:c.val_cap or None]\n    if not train or not val:raise ValueError(\'Empty training/validation selection\')\n    for name in train+val:\n        path=root/\'train_images\'/name\n        if not path.is_file():raise FileNotFoundError(f\'{path}: incomplete download; use explicit smoke IDs only for a code-path check\')\n        with Image.open(path) as image:\n            if image.size!=size:raise ValueError(f\'{name}: image {image.size} does not match {c.variant} {size}\')\n    manifest={\'dataset_variant\':c.variant,\'native_size_wh\':list(size),\'train_image_ids\':train,\n              \'validation_image_ids\':val,\'csv_sha256\':hashlib.sha256(csv_path.read_bytes()).hexdigest(),\n              \'smoke_mode\':bool(smoke),\'scope\':\'code-path smoke; no performance inference\' if smoke else \'capped teaching holdout; not a competition score\',\n              \'training_size_hw\':[c.height,c.width],\'classes\':list(CLASSES),\'split_source\':c.split_manifest or f\'image split seed {c.seed}\'}\n    return records,manifest\n\nclass CloudDataset(Dataset):\n    def __init__(self,c,records,names):self.c=c;self.records=records;self.names=list(names)\n    def __len__(self):return len(self.names)\n    def __getitem__(self,i):\n        name=self.names[i];size=SIZES[self.c.variant];target=(self.c.width,self.c.height)\n        with Image.open(Path(self.c.data_root)/\'train_images\'/name) as im:\n            if im.size!=size:raise ValueError(\'Dataset geometry changed\')\n            x=np.asarray(im.convert(\'RGB\').resize(target,Image.Resampling.BILINEAR),dtype=np.float32)/255\n        masks=[np.asarray(Image.fromarray(decode(self.records[name][label],size)).resize(target,Image.Resampling.NEAREST),dtype=np.float32) for label in CLASSES]\n        return torch.from_numpy(x.transpose(2,0,1).copy()),torch.from_numpy(np.stack(masks)),name\n\nclass TinyCloudNet(nn.Module):\n    """Small encoder/decoder, four independent logits; no BN/dropout in equivalence demos."""\n    def __init__(self):\n        super().__init__();self.encoder=nn.Sequential(nn.Conv2d(3,12,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(12,24,3,padding=1),nn.ReLU())\n        self.decoder=nn.Sequential(nn.Conv2d(24,12,3,padding=1),nn.ReLU(),nn.Conv2d(12,4,1))\n    def forward(self,x):return self.decoder(F.interpolate(self.encoder(x),size=x.shape[-2:],mode=\'bilinear\',align_corners=False))\n\n\ndef sample_losses(logits,y):\n    # Per-image objective: mean BCE + mean channel-wise soft Dice loss.\n    z=logits.float();y=y.float();p=z.sigmoid()\n    bce=F.binary_cross_entropy_with_logits(z,y,reduction=\'none\').mean((1,2,3))\n    dice=(2*(p*y).sum((2,3))+1e-6)/(p.sum((2,3))+y.sum((2,3))+1e-6)\n    return bce+1-dice.mean(1)\n\n\ndef precision_policy(c,device):\n    if c.precision==\'fp16\' and device.type!=\'cuda\':raise ValueError(\'FP16 training demo requires CUDA. Use fp32 or CPU bf16; no silent fallback.\')\n    if c.precision==\'bf16\' and device.type==\'cuda\' and not torch.cuda.is_bf16_supported(including_emulation=False):\n        raise ValueError(\'Native CUDA BF16 unsupported on this GPU; explicitly select fp16 or fp32.\')\n    dtype={\'fp32\':torch.float32,\'fp16\':torch.float16,\'bf16\':torch.bfloat16}[c.precision]\n    def context():return torch.autocast(device.type,dtype=dtype) if c.precision!=\'fp32\' else nullcontext()\n    scaler=torch.amp.GradScaler(device.type,enabled=device.type==\'cuda\' and c.precision==\'fp16\')\n    return context,scaler\n\n\ndef set_seed(seed):\n    random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\n\ndef make_scheduler(optimizer,total,warmup):\n    def factor(step):\n        if warmup and step<warmup:return (step+1)/warmup\n        t=min(1,max(0,(step-warmup)/max(1,total-warmup)))\n        return .1+.9*.5*(1+math.cos(math.pi*t))\n    return torch.optim.lr_scheduler.LambdaLR(optimizer,factor)\n\n\ndef train_epoch(model,loader,optimizer,scheduler,context,scaler,c,device):\n    model.train();iterator=iter(loader);loss_sum=0.;seen=0;updates=0;skipped=0;groups=[]\n    while True:\n        window=list(islice(iterator,c.accumulation))\n        if not window:break\n        count=sum(len(x) for x,_,_ in window);groups.append(count)\n        optimizer.zero_grad(set_to_none=True)\n        for j,(x,y,_) in enumerate(window):\n            x=x.to(device);y=y.to(device);sync=model.no_sync() if hasattr(model,\'no_sync\') and j<len(window)-1 else nullcontext()\n            # DDP no_sync must include forward AND backward.\n            with sync:\n                with context():losses=sample_losses(model(x),y);loss=losses.mean()*(len(x)/count)\n                scaler.scale(loss).backward()\n            loss_sum+=float(losses.detach().sum());seen+=len(x)\n        scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n        did_step=scaler.get_scale()>=old_scale\n        if did_step:scheduler.step();updates+=1\n        else:skipped+=1\n    if not seen:raise ValueError(\'No training batches: reduce batch/world size or provide more training images\')\n    return {\'train_loss\':loss_sum/seen,\'updates\':updates,\'skipped_updates\':skipped,\'window_samples_per_rank\':groups}\n\n\ndef validation_stats(model,loader,context,device):\n    model.eval();stats=torch.zeros(10,dtype=torch.float64,device=device)\n    with torch.inference_mode():\n        for x,y,_ in loader:\n            x=x.to(device);y=y.to(device)\n            with context():z=model(x)\n            loss=sample_losses(z,y);p=z.float().sigmoid()>=.5;g=y.bool()\n            ps=p.sum((2,3));gs=g.sum((2,3));intersection=(p&g).sum((2,3));positive=gs>0;empty=~positive\n            dice=2*intersection/(ps+gs).clamp_min(1)\n            stats[0]+=loss.double().sum();stats[1]+=len(x)\n            stats[2]+=dice[positive].double().sum();stats[3]+=positive.sum()\n            stats[4]+=((ps==0)&empty).sum();stats[5]+=empty.sum();stats[6:]+=positive.sum(0)\n    return stats\n\n\ndef summarize(stats):\n    s=stats.cpu().tolist()\n    return {\'val_loss\':s[0]/s[1] if s[1] else None,\'images\':int(s[1]),\n            \'foreground_dice\':s[2]/s[3] if s[3] else None,\'foreground_pairs\':int(s[3]),\n            \'empty_specificity\':s[4]/s[5] if s[5] else None,\'empty_pairs\':int(s[5]),\n            \'positive_pairs_by_class\':dict(zip(CLASSES,map(int,s[6:])))}\n\n\ndef rng_state():return {\'torch\':torch.get_rng_state(),\'python\':random.getstate(),\n                        \'cuda\':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\ndef restore_rng(s):\n    torch.set_rng_state(s[\'torch\']);random.setstate(s[\'python\'])\n    if s[\'cuda\'] and torch.cuda.is_available():torch.cuda.set_rng_state_all(s[\'cuda\'])\n\ndef atomic_save(state,path):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);temp=path.with_suffix(\'.tmp\');torch.save(state,temp);temp.replace(path)\n\n\ndef run(c=None,resume=None,stop_after=None):\n    c=(c or Config.from_env()).validate();torch.set_num_threads(min(4,os.cpu_count() or 1));set_seed(c.seed)\n    device=torch.device((\'cuda\' if torch.cuda.is_available() else \'cpu\') if c.device==\'auto\' else c.device)\n    records,manifest=prepare(c);out=Path(c.output);out.mkdir(parents=True,exist_ok=True)\n    context,scaler=precision_policy(c,device);model=TinyCloudNet().to(device);optimizer=torch.optim.AdamW(model.parameters(),lr=c.lr)\n    total=c.epochs*math.ceil(math.ceil(len(manifest[\'train_image_ids\'])/c.batch)/c.accumulation)\n    scheduler=make_scheduler(optimizer,total,c.warmup_updates);start=0;best=float(\'inf\');bad=0;history=[]\n    if resume:\n        state=torch.load(resume,map_location=\'cpu\',weights_only=True)\n        if state[\'config\']!=asdict(c) or state[\'manifest\']!=manifest:raise ValueError(\'Resume requires the saved config, split, source hash and schedule horizon; do not change epochs/precision/batch.\')\n        model.load_state_dict(state[\'model\']);optimizer.load_state_dict(state[\'optimizer\']);scheduler.load_state_dict(state[\'scheduler\']);scaler.load_state_dict(state[\'scaler\'])\n        start=state[\'next_epoch\'];best=state[\'best\'];bad=state[\'bad_epochs\'];history=state[\'history\'];restore_rng(state[\'rng\'])\n    valset=CloudDataset(c,records,manifest[\'validation_image_ids\']);vloader=DataLoader(valset,batch_size=c.batch,shuffle=False,num_workers=0)\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\n    print(json.dumps({\'device\':str(device),\'precision\':c.precision,\'gradient_scaling\':scaler.is_enabled(),\'scope\':manifest[\'scope\']}))\n    if bad>=c.patience:print(\'Saved early-stopping patience is exhausted; no further updates.\');return history\n    for epoch in range(start,c.epochs):\n        generator=torch.Generator().manual_seed(c.seed+epoch)\n        loader=DataLoader(CloudDataset(c,records,manifest[\'train_image_ids\']),batch_size=c.batch,shuffle=True,generator=generator,num_workers=0)\n        train=train_epoch(model,loader,optimizer,scheduler,context,scaler,c,device)\n        metrics=summarize(validation_stats(model,vloader,context,device));improved=metrics[\'val_loss\']<best\n        if improved:best=metrics[\'val_loss\'];bad=0\n        else:bad+=1\n        row={\'epoch\':epoch+1,**train,**metrics,\'next_lr\':optimizer.param_groups[0][\'lr\']};history.append(row)\n        state={\'config\':asdict(c),\'manifest\':manifest,\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\n               \'scheduler\':scheduler.state_dict(),\'scaler\':scaler.state_dict(),\'next_epoch\':epoch+1,\'best\':best,\n               \'bad_epochs\':bad,\'history\':history,\'rng\':rng_state()}\n        atomic_save(state,out/\'last.pt\')\n        if improved:atomic_save(state,out/\'best.pt\')\n        (out/\'history.json\').write_text(json.dumps(history,indent=2,allow_nan=False));print(json.dumps(row))\n        if (stop_after and epoch+1>=stop_after) or bad>=c.patience:break\n    return history\n\n\ndef numeric_demo():\n    rows=[]\n    for dtype in [torch.float32,torch.float16,torch.bfloat16]:\n        f=torch.finfo(dtype);values=torch.tensor([1.001,1e5,1e-8],dtype=torch.float32).to(dtype).float().tolist()\n        row={\'dtype\':str(dtype),\'max\':f.max,\'smallest_normal\':f.tiny,\'epsilon_at_one\':f.eps,\'casts_1.001_1e5_1e-8\':values};rows.append(row);print(row)\n    return rows\n\n\ndef benchmark(c,steps=10):\n    """Matched real batch / initial seed / update count. End-to-end training-step timing, not dataloading."""\n    records,m=prepare(c);x,y,_=next(iter(DataLoader(CloudDataset(c,records,m[\'train_image_ids\']),batch_size=c.batch)))\n    device=torch.device((\'cuda\' if torch.cuda.is_available() else \'cpu\') if c.device==\'auto\' else c.device);rows=[]\n    for mode in [\'fp32\',\'fp16\',\'bf16\']:\n        cfg=copy.copy(c);cfg.precision=mode\n        try:context,scaler=precision_policy(cfg,device)\n        except ValueError as e:rows.append({\'precision\':mode,\'status\':\'skipped\',\'reason\':str(e)});continue\n        set_seed(c.seed);model=TinyCloudNet().to(device);optimizer=torch.optim.AdamW(model.parameters(),lr=c.lr);a=x.to(device);b=y.to(device)\n        def step():\n            optimizer.zero_grad(set_to_none=True)\n            with context():loss=sample_losses(model(a),b).mean()\n            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n            return loss\n        for _ in range(3):step()\n        if device.type==\'cuda\':torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()\n        begin=time.perf_counter()\n        for _ in range(steps):loss=step()\n        if device.type==\'cuda\':torch.cuda.synchronize()\n        seconds=time.perf_counter()-begin\n        rows.append({\'precision\':mode,\'device\':str(device),\'steps\':steps,\'seconds\':seconds,\'images_per_second\':steps*len(x)/seconds,\n                     \'peak_allocated_bytes\':torch.cuda.max_memory_allocated() if device.type==\'cuda\' else None,\'final_loss\':float(loss.detach()),\'finite\':bool(torch.isfinite(loss))})\n        del model,optimizer,scaler,a,b\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n    print(json.dumps(rows,indent=2));return rows\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'--demo\',choices=[\'numbers\',\'benchmark\',\'train\'],default=\'train\');parser.add_argument(\'--resume\');parser.add_argument(\'--stop-after\',type=int)\n    args=parser.parse_args()\n    if args.demo==\'numbers\':numeric_demo()\n    elif args.demo==\'benchmark\':benchmark(Config.from_env())\n    else:run(resume=args.resume,stop_after=args.stop_after)\n')
import cloud_training as ct
import torch, copy, json, os
from dataclasses import replace
print(torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
# SETUP ONCE, then use the named demonstration sections below.
# Point at matching train.csv and train_images/ from ONE cloud dataset variant.
cfg = ct.Config(
    data_root=os.environ.get('CLOUD_DATA_ROOT', 'data/bootcamp/clouds_384x576'),
    variant=os.environ.get('CLOUD_DATA_VARIANT', 'resized_384x576'),
    output=os.environ.get('CLOUD_OUTPUT_ROOT', 'runs/cloud_efficiency'),
    smoke_ids=os.environ.get('CLOUD_SMOKE_IMAGE_IDS', ''),
    split_manifest=os.environ.get('CLOUD_SPLIT_MANIFEST', ''),
    precision='fp32', epochs=3, batch=2, accumulation=2)
records, manifest = ct.prepare(cfg)
print(json.dumps(manifest, indent=2))
train_set = ct.CloudDataset(cfg, records, manifest['train_image_ids'])
x, y, image_id = train_set[0]
print(image_id, 'input', tuple(x.shape), 'target', tuple(y.shape))
assert x.shape == (3, 64, 96) and y.shape == (4, 64, 96)
assert set(y.unique().tolist()) <= {0., 1.}


## 01 · FP16 versus BF16: range is not precision
Predict the cast of 1.001, 100000 and 1e-8 before running. Both low-precision
formats use 16 bits. FP16 has 5 exponent + 10 stored fraction bits; BF16 has 8 + 7.
`finfo.tiny` means smallest **normal**, not smallest subnormal. Tensor casts
illustrate representation; they do not emulate an entire AMP training run.


In [ ]:
ct.numeric_demo()


## 02 · AMP on one real cloud batch
Start from the same random seed, batch and number of updates for every policy.
Read the implementation before benchmarking: autocast wraps forward/loss;
backward is outside. Our per-image BCE+Dice reduces logits in FP32.

Three warmup updates precede the measured training steps. CUDA timers synchronize
before and after measurement; peak memory counts allocated tensors, not all GPU
reserved/device memory. A tiny model can be overhead-bound. No fixed speedup is
expected. Unsupported policies are labelled skipped. CPU timings are not CUDA
Tensor Core evidence. Compare finite loss and validate quality separately.


In [ ]:
import inspect
print(inspect.getsource(ct.precision_policy))
benchmark_rows = ct.benchmark(cfg, steps=10)


## 03 · Accumulation, including an uneven last microbatch
Compare gradients for the SAME real samples in a physical batch and microbatches.
For a window of N images, weight each microbatch mean loss by B_i/N, not always
1/K. This handles smaller final microbatches. We compare gradients before clipping
or updating. The model has no BatchNorm/dropout and the loss is per-image separable;
those restrictions matter. Roundoff means allclose, not identical bits.


In [ ]:
from torch.utils.data import DataLoader
ct.set_seed(29)
x, y, ids = next(iter(DataLoader(train_set, batch_size=min(5, len(train_set)))))
a = ct.TinyCloudNet(); b = copy.deepcopy(a)
ct.sample_losses(a(x), y).mean().backward()
for xx, yy in zip(x.split(2), y.split(2)):
    (ct.sample_losses(b(xx), yy).mean() * (len(xx)/len(x))).backward()
errors = [(pa.grad-pb.grad).abs().max().item() for pa,pb in zip(a.parameters(),b.parameters())]
assert all(torch.allclose(pa.grad,pb.grad,atol=2e-6,rtol=2e-5) for pa,pb in zip(a.parameters(),b.parameters()))
print('Samples:', len(x), 'microbatch sizes:', [len(t) for t in x.split(2)])
print('Maximum gradient difference:', max(errors))
print(inspect.getsource(ct.train_epoch))


## 04 · Interrupt, resume, then compare with an uninterrupted run
Use fresh output folders for a new experiment. The complete schedule horizon is
three epochs from the beginning; interruption after epoch one does not change it.
The saved checkpoint restores model, optimizer, scheduler, scaler, next epoch,
best value, patience, split/config/source fingerprint and RNG state.

This is epoch-boundary resume with deterministic sample order, workers=0 and no
augmentation. Exact arbitrary mid-epoch/distributed/hardware replay is not promised.
The monitored value is validation loss; best.pt and last.pt serve different purposes.


In [ ]:
resume_cfg = replace(cfg, output=str(Path(cfg.output)/'resume_demo'))
reference_cfg = replace(cfg, output=str(Path(cfg.output)/'reference_demo'))
ct.run(resume_cfg, stop_after=1)
checkpoint = Path(resume_cfg.output)/'last.pt'
saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
print('Restoring keys:', list(saved), 'next epoch:', saved['next_epoch'])
resumed_history = ct.run(resume_cfg, resume=checkpoint)
reference_history = ct.run(reference_cfg)
r = torch.load(checkpoint, map_location='cpu', weights_only=True)['model']
u = torch.load(Path(reference_cfg.output)/'last.pt', map_location='cpu', weights_only=True)['model']
resume_error = max((r[k]-u[k]).abs().max().item() for k in r)
print('Resumed vs uninterrupted maximum weight difference:', resume_error)
if cfg.device=='cpu' or not torch.cuda.is_available():
    assert all(torch.allclose(r[k],u[k],atol=1e-7,rtol=1e-6) for k in r)


## 05 · Inspect the schedule and validation evidence
The scheduler steps after a successful optimizer update, not every microbatch.
A skipped FP16 update does not advance it. Print a fresh schedule without training.
Then read REAL held-out loss, foreground-pair Dice, empty-pair specificity and
class support from the measured run. An absent denominator is null, not success.
These are fixed 64×96 teaching-grid diagnostics, not official cloud or industrial
competition scores. Best weights need not be last weights.


In [ ]:
probe = torch.optim.SGD(ct.TinyCloudNet().parameters(), lr=.001)
schedule = ct.make_scheduler(probe, total=12, warmup=2)
lrs=[]
for update in range(12):
    lrs.append(probe.param_groups[0]['lr']); probe.step(); schedule.step()
print('Learning rates used:', lrs)
print('Actual final validation record:', resumed_history[-1])
print('Best checkpoint exists:', (Path(resume_cfg.output)/'best.pt').is_file())


## Reflect
What did AMP change? Did memory, time and quality move together? Why does an
incomplete accumulation window need its actual denominator? What state was needed
to resume? Choose one next experiment and keep split, data order, model start,
effective batch and validation protocol fixed. DDP is a separate optional notebook.
